
# Geneformer Foundation Model Analysis of PBMC Single-Cell RNA-seq

## 1. Project overview

This project explores pretrained Geneformer V1 cell embeddings from a PBMC single-cell RNA-seq dataset. The analysis examines embedding structure, compares Geneformer embeddings with a reconstructed PCA baseline, and evaluates neighborhood consistency against existing Leiden cluster labels.

## 2. Objectives

- Extract cell embeddings using a pretrained foundation model.
- Visualize embeddings using UMAP.
- Explore marker-gene-supported cell-type annotations.
- Compare representations using silhouette scores.
- Evaluate nearest-neighbor consistency and document limitations.

## 3. Dataset

- Number of cells: 2,671
- Number of Leiden clusters: 6
- Geneformer embedding dimensions: 256

## 4. Workflow

1. Prepare count data and Ensembl gene identifiers.
2. Tokenize cells with Geneformer V1.
3. Extract pretrained cell embeddings.
4. Generate a UMAP visualization.
5. Explore marker genes and annotate Leiden clusters.
6. Compare silhouette scores.
7. Evaluate nearest-neighbor consistency.

## 5. Preliminary results

| Representation | Dimensions | Cosine silhouette |
|---|---:|---:|
| Geneformer V1 | 256 | 0.1015 |
| Reconstructed PCA baseline | 30 | 0.3044 |

Mean same-annotation fraction among five nearest neighbors: 0.8958.

## 6. Preliminary cluster annotations

| Leiden cluster | Cells | Annotation |
|---|---:|---|
| 0 | 1,183 | T cells |
| 1 | 638 | Myeloid cells / monocytes |
| 2 | 455 | NK / cytotoxic lymphocytes |
| 3 | 347 | B cells |
| 4 | 33 | Antigen-presenting cells; unresolved |
| 5 | 15 | Platelets |

## 7. Limitations

The PCA baseline was reconstructed and may not reproduce the original clustering representation. Silhouette scores measure agreement with existing labels, not biological correctness. Neighborhood consistency is not independent biological validation because the annotations derive from Leiden clusters. Small clusters and the unresolved antigen-presenting group require cautious interpretation.

## 8. Reproducibility

The embeddings and analysis outputs were generated in Google Colab using Geneformer V1 and Python single-cell analysis tools. The notebook should be extended with executable preprocessing, tokenization, embedding extraction, and evaluation cells to provide a fully reproducible workflow.


## 9. Load saved analysis outputs

This section loads existing project outputs. It does not rerun Geneformer or regenerate the embeddings.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..")
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

embedding_file = RESULTS_DIR / "pbmc3k_geneformer_v1.csv"
umap_file = RESULTS_DIR / "pbmc3k_geneformer_v1_umap.csv"
annotations_file = RESULTS_DIR / "pbmc_leiden_cell_annotations.csv"
summary_file = RESULTS_DIR / "geneformer_analysis_summary.csv"

umap_df = pd.read_csv(umap_file)
annotations = pd.read_csv(annotations_file)
summary = pd.read_csv(summary_file)

print("UMAP rows and columns:", umap_df.shape)
print("Annotation table shape:", annotations.shape)
print("\nAnalysis summary:")
display(summary)


## 10. Validate the saved outputs

Check for missing UMAP coordinates, inspect cluster counts, and confirm that annotations cover the six Leiden clusters.


In [ ]:
required_umap_columns = {"UMAP1", "UMAP2", "leiden"}
assert required_umap_columns.issubset(umap_df.columns), "Missing expected UMAP columns"
assert not umap_df[["UMAP1", "UMAP2"]].isna().any().any(), "Missing UMAP coordinates"

cluster_counts = (
    umap_df["leiden"].astype(str)
    .value_counts()
    .sort_index()
    .rename_axis("leiden")
    .reset_index(name="cell_count")
)

annotation_check = annotations.copy()
annotation_check["leiden"] = annotation_check["leiden"].astype(str)
cluster_counts["leiden"] = cluster_counts["leiden"].astype(str)

validation = cluster_counts.merge(
    annotation_check[["leiden", "annotation"]],
    on="leiden",
    how="left",
    validate="one_to_one",
)

assert validation["annotation"].notna().all(), "Some clusters lack annotations"

print("Total cells represented in UMAP:", len(umap_df))
print("Number of clusters:", umap_df["leiden"].nunique())
display(validation)


## 11. Recreate the saved UMAP visualization

This plot uses the previously calculated UMAP coordinates. It does not rerun UMAP or the Geneformer model.


In [ ]:
import matplotlib.pyplot as plt

plot_df = umap_df.copy()
plot_df["leiden"] = plot_df["leiden"].astype(str)

fig, ax = plt.subplots(figsize=(8, 6))
for cluster, group in plot_df.groupby("leiden", sort=True):
    ax.scatter(
        group["UMAP1"],
        group["UMAP2"],
        s=8,
        alpha=0.7,
        label=f"Cluster {cluster}",
        rasterized=True,
    )

ax.set_title("PBMC cells in Geneformer V1 embedding space")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(title="Leiden", markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()
